In [ ]:
import pandas as pd

df = pd.read_pickle("../scripts/data/checkpoint_after_cleaning.pkl")

df = df.rename(columns={'UTC': 'utc'})

df.head(10)

### Analyze missing data for each feature

In [ ]:
pollutant = ['PM2.5', 'OZONE', 'NO2', 'SO2', 'CO', 'PM10']
weather_features = ['temperature_2m', 'precipitation', 'weather_code', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'relative_humidity_2m']

all_features = ['latitude', 'longitude', 'utc'] + weather_features + pollutant
existing_features = [f for f in all_features if f in df.columns]

missing_features = pd.DataFrame({
    'Feature': existing_features,
    'MissingPercent': (df[existing_features].isnull().sum() / len(df)) * 100,
})

missing_features_summary = missing_features.sort_values('MissingPercent', ascending=False)

print(missing_features_summary)

### Identify which pollutants are measured at each location

In [ ]:
pollutants = ['PM2.5', 'OZONE', 'NO2', 'SO2', 'CO', 'PM10']

locations = df.groupby(['latitude', 'longitude'])

for (lat, lon), group in locations:
    measured = []
    for pollutant in pollutants:
        if group[pollutant].notna().any():
            measured.append(pollutant)

    # Get location info from first row of this location group
    county = group['County Name'].iloc[0]
    sitename = group['sitename'].iloc[0]
    aqscode = group['fullaqscode'].iloc[0]

    print(f"{county}, {sitename}, {aqscode}, ({lat:.4f}, {lon:.4f}) → {measured}")

###  Drop sites that do not measure label pm2.5.

In [ ]:
pm25_by_location = df.groupby(['latitude', 'longitude'])['PM2.5'].apply(lambda x: x.notna().any())

valid_locations = pm25_by_location[pm25_by_location].index

print(f"\nSites with PM2.5 sensor: {len(valid_locations)}")

for (lat, lon), group in df.groupby(['latitude', 'longitude']):
    if (lat, lon) in valid_locations:
        county = group['County Name'].iloc[0]
        sitename = group['sitename'].iloc[0]
        aqscode = group['fullaqscode'].iloc[0]
        print(f"{county}, {sitename}, {aqscode}, ({lat:.4f}, {lon:.4f})")

#### keep only rows from valid locations

In [ ]:
df = df[df.set_index(['latitude', 'longitude']).index.isin(valid_locations)].reset_index(drop=True)

df.head(10)

## Check Missing Data Again

In [ ]:

missing_features = pd.DataFrame({
    'Feature': existing_features,
    'MissingPercent': (df[existing_features].isnull().sum() / len(df)) * 100,
})

missing_features_summary = missing_features.sort_values('MissingPercent', ascending=False)

print(missing_features_summary)

### Convert UTC to datetime

In [ ]:
import numpy as np

# convert UTC to date/time
df['utc'] = pd.to_datetime(df['utc'], errors='coerce', utc=True)
# get hour of day
df['hr'] = df['utc'].dt.hour
# sine and cosine transformation for hour
df['sin_hr'] = np.sin(2 * np.pi * df['hr'] / 24)
df['cos_hr'] = np.cos(2 * np.pi * df['hr'] / 24)
# sine and cosine transformation for days
df['day'] = df['utc'].dt.dayofweek
df['sin_day'] = np.sin(2 * np.pi * df['day'] / 7)
df['cos_day'] = np.cos(2 * np.pi * df['day'] / 7)

df.head(10)

### List sites that have PM2.5 sensor

In [ ]:
pollutants = ['PM2.5', 'OZONE', 'NO2', 'SO2', 'CO', 'PM10']

print("\nsites with PM2.5:")
for (lat, lon), group in df.groupby(['latitude', 'longitude']):

    # location info
    county = group['County Name'].iloc[0]
    sitename = group['sitename'].iloc[0]
    aqscode = group['fullaqscode'].iloc[0]

    # get pollutants this site measures
    measured = []
    for pollutant in pollutants:
        if pollutant in df.columns and group[pollutant].notna().any():
            measured.append(pollutant)

    print(f"{county}, {sitename}, {aqscode}, ({lat:.4f}, {lon:.4f}) → {measured}")


### Select a site to analyze

In [ ]:
df['fullaqscode'] = df['fullaqscode'].astype(str)

# Selection
# 60371201
# 60371103
selected_site = ['320030073']

df = df[df['fullaqscode'].isin(selected_site)].copy()

print(f"Selected site: {df['sitename'].iloc[0]}")

In [ ]:
df.head(10)

## Check Missing Data And drop features with too much missing data

In [ ]:
import matplotlib.pyplot as plt

pollutants_features = ['OZONE', 'NO2', 'SO2', 'CO', 'PM10']
weather_features = ['temperature_2m', 'precipitation', 'weather_code', 'wind_speed_10m',
                    'wind_direction_10m', 'wind_gusts_10m', 'relative_humidity_2m']

percentage_threshold = 50

all_features = pollutants_features + weather_features

# some features may not exist at this site
existing_features = [f for f in all_features if f in df.columns]

missing = pd.DataFrame({'Feature': existing_features, 'MissingPercent': [(df[f].isnull().sum() / len(df)) * 100 for f in existing_features]})

features_to_drop = missing[missing['MissingPercent'] > percentage_threshold]['Feature'].tolist()
features_to_keep = missing[missing['MissingPercent'] <= percentage_threshold]['Feature'].tolist()

print(missing.sort_values('MissingPercent', ascending=False).to_string(index=False))

# Drop features with too much missing data
if features_to_drop:
    df = df.drop(columns=features_to_drop)


plt.figure(figsize=(10, 6))
sorted_summary = missing.sort_values('MissingPercent', ascending=True)
colors = ['red' if pct > percentage_threshold else 'green' for pct in sorted_summary['MissingPercent']]

plt.barh(sorted_summary['Feature'], sorted_summary['MissingPercent'], color=colors)
plt.axvline(x=percentage_threshold, color='black', linestyle='--', linewidth=2,
            label=f'Threshold ({percentage_threshold}%)')
plt.xlabel('Missing Data (%)')
plt.ylabel('Feature')
plt.title(f'Missing Data by Feature - Site: {df["sitename"].iloc[0]}')
plt.legend()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Check PM2.5 missing by year

In [ ]:
df['year'] = df['utc'].dt.year

pm25_by_year = df.groupby('year').agg({'PM2.5': [lambda x: (x.isnull().sum() / len(x)) * 100]})

pm25_by_year.columns = ['PM2.5_Missing_%']

print(pm25_by_year)

### Interpolate missing feature data

In [ ]:
df = df.sort_values('utc').reset_index(drop=True)


pollutant_features = [col for col in ['OZONE', 'NO2', 'SO2', 'CO', 'PM10'] if col in df.columns]

weather_features = [col for col in ['temperature_2m', 'precipitation', 'weather_code', 'wind_speed_10m', 'wind_direction_10m',
                                    'wind_gusts_10m', 'relative_humidity_2m'] if col in df.columns]

feature_cols = pollutant_features + weather_features


for col in feature_cols:

    # Flag where this feature was originally missing
    df[col + '_missing_flag'] = df[col].isna().astype(int)

    df[col] = df[col].interpolate(method='linear', limit=6, limit_direction='both')

    df[col] = df[col].fillna(df[col].median())

df.head(10)

### Interpolate PM2.5, but keep track of original data so we can test on only original labels

In [ ]:
# original PM2.5
df['PM2.5_real'] = df['PM2.5']


df['PM2.5_interpolated'] = (df['PM2.5'].interpolate(method='linear', limit_direction='both'))

df = df[~df['PM2.5_interpolated'].isna()].reset_index(drop=True)

df['is_real_label'] = df['PM2.5_real'].notna()

### Check missing data after interpolation

In [ ]:
pollutants = ['PM2.5', 'OZONE', 'NO2', 'SO2', 'CO', 'PM10']
weather_features = ['temperature_2m', 'precipitation', 'weather_code', 'wind_speed_10m',
                    'wind_direction_10m', 'wind_gusts_10m', 'relative_humidity_2m']

all_features = pollutants + weather_features
existing_features = [f for f in all_features if f in df.columns]

missing_summary = pd.DataFrame({'Feature': existing_features, 'MissingPercent': [(df[f].isnull().sum() / len(df)) * 100 for f in existing_features]})

print("Missing data after interpolation:")
print(missing_summary.sort_values('MissingPercent', ascending=False).to_string(index=False))


In [ ]:
# Check PM2.5 missing by year
df['year'] = df['utc'].dt.year

pm25_by_year = df.groupby('year').agg({'PM2.5_interpolated': [lambda x: (x.isnull().sum() / len(x)) * 100]})

pm25_by_year.columns = ['PM2.5_Missing_%']

print(pm25_by_year)

